# Business entity resolution: full run on Kaggle

**normalize → probe → blocking (train + test, full S1 index) → features → train → prefilter → tune → predict → validate**

Built for **Save Version → Save & Run All (Commit)**: it runs unattended in the background, no browser needed.
- Settings: **Internet ON** (only for `pip install`; no external data is used), CPU session is enough.
- Inputs: the challenge data (`train_source1.tsv` is auto-detected under `/kaggle/input`), the code dataset
  containing the `business_entity_resolution/` folder, and optionally **the output of an earlier run of this
  notebook** (see "Resume" below).
- Everything is written under `/kaggle/working`: `output/` (submission TSVs, `variants/`, pair probabilities),
  `cache/` (checkpoints), `run.log` (full progress log).
- The last cell prints the OOF macro F0.5, the distractor-weighted thresholds and the validator verdicts.

## Step 1: locate the inputs and set paths
Finds the challenge data and the code dataset, copies the code to `/kaggle/working`, and sets the data / cache /
output directories through the `BER_*` environment variables read by `config.py`.

In [ ]:
import glob, os, shutil, subprocess, sys

# Auto-detect the dataset: the directory whose train/ holds train_source1.tsv.
DEFAULT_DATA_DIR = "/kaggle/input/datasets/asadahmedshaik/amlc26-data/student_resource/dataset"
found_data = sorted(glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True))
DATA_DIR = os.path.dirname(os.path.dirname(found_data[0])) if found_data else DEFAULT_DATA_DIR
assert os.path.exists(f"{DATA_DIR}/test/test_source1.tsv"), f"test files not found next to {DATA_DIR}/train"
WORK = "/kaggle/working"
CACHE_DIR, OUTPUT_DIR, LOG = f"{WORK}/cache", f"{WORK}/output", f"{WORK}/run.log"
WORK_CODE = f"{WORK}/business_entity_resolution"

# Locate the code (inputs are read-only, so copy it into /kaggle/working).
hits = sorted(glob.glob("/kaggle/input/**/business_entity_resolution/src/run_all.py", recursive=True)
              + glob.glob(f"{WORK}/**/business_entity_resolution/src/run_all.py", recursive=True))
assert hits, "business_entity_resolution/src/run_all.py not found under /kaggle/input or /kaggle/working"
found = os.path.dirname(os.path.dirname(hits[0]))
if os.path.abspath(found) != WORK_CODE:
    shutil.copytree(found, WORK_CODE, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
CODE_DIR = WORK_CODE

# The official validator (utils/validate_submission.py), if it is anywhere in the inputs.
validators = sorted(glob.glob("/kaggle/input/**/utils/validate_submission.py", recursive=True))
os.environ.update(BER_DATA_DIR=DATA_DIR, BER_CACHE_DIR=CACHE_DIR, BER_OUTPUT_DIR=OUTPUT_DIR,
                  BER_VALIDATOR=validators[0] if validators else "", PYTHONIOENCODING="utf-8")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("code:     ", CODE_DIR)
print("data:     ", DATA_DIR)
print("validator:", validators[0] if validators else "NOT FOUND (validation will be skipped)")

## Resume from an earlier run (optional)
If the output of an earlier version of this notebook is attached as an input, its `cache/` (normalised data,
blocking, features, trained models) is copied to `/kaggle/working/cache` with timestamps preserved, and the run
uses `--trust-cache`: every finished stage is skipped and only the missing ones (e.g. `prefilter`, `tune`, `predict`) run; a stage whose inputs were rebuilt in this run reruns too.
Only do this when the pipeline code of those stages has not changed since that run.

In [ ]:
prev = sorted(glob.glob("/kaggle/input/**/cache/model/meta.json", recursive=True))
RESUMED = False
if prev and not os.path.exists(f"{CACHE_DIR}/model/meta.json"):
    src = os.path.dirname(os.path.dirname(prev[0]))          # .../cache
    print("copying earlier cache:", src, "->", CACHE_DIR)
    shutil.copytree(src, CACHE_DIR, dirs_exist_ok=True, copy_function=shutil.copy2,
                    ignore=shutil.ignore_patterns("sample_*"))
    RESUMED = True
elif os.path.exists(f"{CACHE_DIR}/model/meta.json"):
    RESUMED = True                                           # interactive re-run: cache already in place
print("resume from earlier cache:", RESUMED)
!du -sh {CACHE_DIR}/* 2>/dev/null | head -20

## Step 3: install the CPU requirements
`requirements.txt` without the torch / transformers / tokenizers pins (those are only needed by the GPU
cross-encoder notebooks).

In [ ]:
!grep -v -E "^(torch|transformers|tokenizers)==" {CODE_DIR}/requirements.txt > /tmp/req_cpu.txt
!pip install -q -r /tmp/req_cpu.txt 2>&1 | tail -3   # CE pins are for the GPU stage only
!nproc && free -g && df -h /kaggle/working | tail -1

## Run settings
- `RUN_SMOKE`: 50k-S1 train sample through training first (~10 min), to fail fast on environment problems.
- `RUN_PROBE`: blocking scale probe on the full test index (~10 min); skipped automatically when resuming.
- `EXTRA_ARGS`: blocking knobs, e.g. `"--pre_cap 100"`; empty = defaults (M=1, pre_cap 150, block_cap 2000).
- The final threshold of `matching_results.tsv` is `config.FINAL_THRESHOLD` (0.79); the variant
  submissions in `output/variants/` use the thresholds chosen by `tune_threshold.py`.
- `candidate_pairs.tsv` is the pre-filtered set the matcher scored (`config.PREFILTER`, `config.PREFILTER_CUTOFF`).
- Optional, unused experiment (run 5): add `train_pw models` to the steps for a pairwise-only matcher and the
  base/pairwise mean (`output/variants/matching_results_{pairwise,ensemble}_t*.tsv`).

In [ ]:
RUN_SMOKE = False
RUN_PROBE = not RESUMED
EXTRA_ARGS = "--trust-cache" if RESUMED else ""

def run_all(args):
    """Run run_all.py, streaming its output here (it also appends everything to run.log)."""
    cmd = f"{sys.executable} -u {CODE_DIR}/src/run_all.py --log {LOG} {args} {EXTRA_ARGS}"
    print(">>>", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode

if RUN_SMOKE:
    run_all("--sample 50000 --steps normalize blocking features train")
steps = "normalize " + ("probe " if RUN_PROBE else "") + "blocking features train prefilter tune predict validate"
rc = run_all(f"--steps {steps}")
print("run_all exit code:", rc)

## Step 5: final report
The summary at the end of `run.log` (OOF macro F0.5, thresholds, blocking recall, validator verdict) and the
output files.

In [ ]:
# Final report (also at the end of run.log)
log = open(LOG, encoding="utf-8").read() if os.path.exists(LOG) else ""
print(log[log.rfind("SUMMARY") - 72:] if "SUMMARY" in log else "no summary in run.log: see the output above")
print(subprocess.run(f"ls -la {OUTPUT_DIR} {OUTPUT_DIR}/variants; head -3 {OUTPUT_DIR}/matching_results.tsv; "
                     f"du -sh {CACHE_DIR}", shell=True, capture_output=True, text=True).stdout)